# STEP Files

Clear the OCP CAD Viewer first, then refresh and open the canonical `type2_scene.step` artifact in VS Code. Active plate-stack scenes include stub overhang bodies; metadata-only port sheets are still reconstructed on the AEDT side, not exported into STEP.


In [96]:
from __future__ import annotations



import gc
import sys

from IPython import get_ipython
from ocp_vscode import show_clear


def _clear_notebook_state() -> int:
    ip = get_ipython()
    if ip is None:
        raise RuntimeError("This cell must run inside an IPython kernel")

    namespace = ip.user_ns
    keep_names = {
        "__name__",
        "__doc__",
        "__package__",
        "__loader__",
        "__spec__",
        "__builtins__",
        "__builtin__",
        "get_ipython",
        "In",
        "Out",
        "_ih",
        "_oh",
        "_dh",
        "exit",
        "quit",
    }
    for name in tuple(namespace):
        if name in keep_names:
            continue
        del namespace[name]
        

    if "Out" in namespace:
        out_cache = namespace["Out"]
        if isinstance(out_cache, dict):
            out_cache.clear()
        

    for last_name in ("_", "__", "___"):
        if last_name in namespace:
            namespace[last_name] = None
        
    import sys
    for module_name in tuple(sys.modules):
        if module_name == "peetsfea" or module_name.startswith("peetsfea."):
            del sys.modules[module_name]
        import sys

    if "matplotlib.pyplot" in sys.modules:
        import matplotlib.pyplot as plt
        plt.close("all")
        
    from ocp_vscode import show_clear
    show_clear()
    import gc
    return gc.collect()


_collected = _clear_notebook_state()

In [97]:

VIEW_INDEX = 19
BUILD_W_GUI = False
print(f"notebook state cleared; gc collected: {_collected}")
print(f"view index: {VIEW_INDEX}")
del _collected


notebook state cleared; gc collected: 4406
view index: 19


In [98]:
from __future__ import annotations

from pathlib import Path
import subprocess
import sys

import build123d as bd
from ocp_vscode import Camera, show, show_clear


def require_repo_root() -> Path:
    result = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    )
    root_text = result.stdout.strip()
    if not root_text:
        raise RuntimeError("git rev-parse --show-toplevel returned empty stdout")
    repo_root = Path(root_text).resolve()
    pyproject_path = repo_root / "pyproject.toml"
    if not pyproject_path.is_file():
        raise FileNotFoundError(f"repo root is missing pyproject.toml: {pyproject_path}")
    return repo_root


REPO_ROOT = require_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from peetsfea.backend.pyaedt import type2_step_import_style as aedt_style
from peetsfea.type2_sampled import manifest_entry_for_sample_index
from peetsfea.type2_step_export import export_type2_step_artifacts


TYPE2_FIXED_SOURCE_TOML_PATH = REPO_ROOT / "examples" / "type2_fixed.toml"
TYPE2_FIXED_OUTPUT_DIR = REPO_ROOT / "run" / "step" / "type2"
TYPE2_FIXED_LEDGER_PATH = TYPE2_FIXED_OUTPUT_DIR / "type2_step_ledger.json"
TYPE2_SAMPLED_MANIFEST_PATH = REPO_ROOT / "run" / "sampled" / "type2" / "manifest.json"


def alpha_from_transparency(transparency: float) -> float:
    if transparency < 0.0 or transparency > 1.0:
        raise ValueError(f"viewer transparency must be in [0, 1] (actual={transparency})")
    return 1.0 - transparency


_NON_MODEL_STYLE = (
    aedt_style._NON_MODEL_COLOR,
    alpha_from_transparency(aedt_style._NON_MODEL_TRANSPARENCY),
)
_PCB_STYLE = (
    aedt_style._TX_PCB_COLOR,
    alpha_from_transparency(aedt_style._TX_PCB_TRANSPARENCY),
)
_COPPER_STYLE = (
    aedt_style._TX_COPPER_COLOR,
    alpha_from_transparency(aedt_style._TX_COPPER_TRANSPARENCY),
)
_FERRITE_STYLE = (
    aedt_style._TX_UNDERLAY_FERRITE_COLOR,
    alpha_from_transparency(aedt_style._TX_UNDERLAY_FERRITE_TRANSPARENCY),
)
_PET_PSA_STYLE = (
    aedt_style._TX_UNDERLAY_PET_PSA_COLOR,
    alpha_from_transparency(aedt_style._TX_UNDERLAY_PET_PSA_TRANSPARENCY),
)
_AIR_STYLE = (
    aedt_style._TX_UNDERLAY_AIR_COLOR,
    alpha_from_transparency(aedt_style._TX_UNDERLAY_AIR_TRANSPARENCY),
)


def viewer_style_from_label(label: str) -> tuple[tuple[int, int, int], float]:
    if label.startswith(("tx_copper", "rx_copper", "tx_bridge", "rx_bridge", "tx_stub", "rx_stub")):
        return _COPPER_STYLE
    if label.startswith(("tx_pcb", "rx_pcb")):
        return _PCB_STYLE
    if label.startswith(aedt_style._UNDERLAY_FERRITE_NAME_PREFIXES):
        return _FERRITE_STYLE
    if label.startswith(aedt_style._UNDERLAY_PET_PSA_NAME_PREFIXES):
        return _PET_PSA_STYLE
    if label.startswith(aedt_style._UNDERLAY_AIR_NAME_PREFIXES):
        return _AIR_STYLE
    return _NON_MODEL_STYLE


def child_shapes(shape: bd.Shape) -> list[bd.Shape]:
    children = tuple(shape.children)
    if children:
        return list(children)
    return [shape]


def viewer_payload_for_shape(shape: bd.Shape, *, fallback_name: str) -> tuple[list[bd.Shape], list[str], list[tuple[int, int, int]], list[float]]:
    entries = child_shapes(shape)
    cad_objs: list[bd.Shape] = []
    names: list[str] = []
    colors: list[tuple[int, int, int]] = []
    alphas: list[float] = []
    for index, entry in enumerate(entries):
        entry_label = entry.label if isinstance(entry.label, str) and entry.label != "" else f"{fallback_name}_{index}"
        color, alpha = viewer_style_from_label(entry_label)
        cad_objs.append(entry)
        names.append(entry_label)
        colors.append(color)
        alphas.append(alpha)
    return (cad_objs, names, colors, alphas)


print(f"repo root: {REPO_ROOT}")
print(f"view index: {VIEW_INDEX}")
print(f"fixed type2 TOML: {TYPE2_FIXED_SOURCE_TOML_PATH}")
print(f"sample manifest: {TYPE2_SAMPLED_MANIFEST_PATH}")
show_clear()
print("viewer cleared")



repo root: /home/harry/Projects/PythonProjects/worktrees/peetsfea-dev2
view index: 19
fixed type2 TOML: /home/harry/Projects/PythonProjects/worktrees/peetsfea-dev2/examples/type2_fixed.toml
sample manifest: /home/harry/Projects/PythonProjects/worktrees/peetsfea-dev2/run/sampled/type2/manifest.json
viewer cleared


## Refresh and show scene STEP

This notebook clears the viewer first, refreshes the fixed `type2_scene.step` artifact, and opens that single scene file with a reset camera. Plate-stack STEP scenes include shoe-fill and stub bodies; reconstructed port sheets remain metadata-only.


In [99]:
if VIEW_INDEX == -1:
    fixed_ledger = export_type2_step_artifacts(
        toml_path=TYPE2_FIXED_SOURCE_TOML_PATH,
        output_dir=TYPE2_FIXED_OUTPUT_DIR,
        ledger_path=TYPE2_FIXED_LEDGER_PATH,
        seed=0,
    )
    scene_step_path = Path(fixed_ledger["scene_step_path"])
    print("mode: fixed example")
    print(f"source TOML: {TYPE2_FIXED_SOURCE_TOML_PATH}")
    print(f"scene STEP: {scene_step_path}")
    print(f"ledger JSON: {TYPE2_FIXED_LEDGER_PATH}")
else:
    selected_entry = manifest_entry_for_sample_index(
        TYPE2_SAMPLED_MANIFEST_PATH,
        sample_index=VIEW_INDEX,
    )
    scene_step_path = Path(selected_entry["scene_step_path"])
    if not scene_step_path.is_file():
        raise FileNotFoundError(
            "sampled scene STEP not found for notebook view index "
            f"(index={VIEW_INDEX}, design_id={selected_entry['design_id']}, path={scene_step_path})"
        )
    print("mode: sampled manifest")
    print(f"manifest: {TYPE2_SAMPLED_MANIFEST_PATH}")
    print(f"sample index: {selected_entry['sample_index']}")
    print(f"design_id: {selected_entry['design_id']}")
    print(f"seed: {selected_entry['seed']}")
    print(f"retry: {selected_entry['retry_number']}")
    print(f"scene STEP: {scene_step_path}")

shown_step = bd.import_step(scene_step_path)
cad_objs, names, colors, alphas = viewer_payload_for_shape(shown_step, fallback_name="type2_scene")
show(
    *cad_objs,
    names=names,
    colors=colors,
    alphas=alphas,
    transparent=True,
    reset_camera=Camera.RESET,
)



mode: sampled manifest
manifest: /home/harry/Projects/PythonProjects/worktrees/peetsfea-dev2/run/sampled/type2/manifest.json
sample index: 19
design_id: s000019_942b_575c_0
seed: 19
retry: 0
scene STEP: /home/harry/Projects/PythonProjects/worktrees/peetsfea-dev2/run/sampled/type2/s000019_942b_575c_0/type2_scene.step
cccccc++++c++ccc+++++++++


In [100]:
import subprocess

from peetsfea.type2_sampled import manifest_entry_for_sample_index


if "BUILD_W_GUI" not in globals():
    raise RuntimeError("Run the notebook state cell first so BUILD_W_GUI is defined")
if not isinstance(BUILD_W_GUI, bool):
    raise TypeError(f"BUILD_W_GUI must be bool (actual={type(BUILD_W_GUI).__name__})")
print(f"BUILD_W_GUI: {BUILD_W_GUI}")

if BUILD_W_GUI:
    if VIEW_INDEX < 0:
        raise RuntimeError("GUI debug build requires sampled manifest VIEW_INDEX >= 0")
    selected_entry = manifest_entry_for_sample_index(
        TYPE2_SAMPLED_MANIFEST_PATH,
        sample_index=VIEW_INDEX,
    )
    design_id = selected_entry["design_id"]
    command = [
        str(REPO_ROOT / ".venv" / "bin" / "python"),
        str(REPO_ROOT / "entry" / "build.py"),
        "--debug",
        "--manifest",
        str(TYPE2_SAMPLED_MANIFEST_PATH),
        "--design-id",
        design_id,
    ]
    print(f"manifest: {TYPE2_SAMPLED_MANIFEST_PATH}")
    print(f"debug design_id: {design_id}")
    print("GUI debug build delegated to entry/build.py")
    subprocess.run(command, cwd=REPO_ROOT / "run", check=True)
else:
    print("BUILD_W_GUI is False; GUI build skipped")


BUILD_W_GUI: False
BUILD_W_GUI is False; GUI build skipped
